# Dependency Collocation Workflow

This notebook is a runnable recipe for dependency-path and syntactic collocation counts. The reusable pipeline functions stay in `dep_colloc/`; notebook-local functions are diagnostics or export helpers.

## 1. Configuration

The parser expects the current seven-column token format with a morphology/FEATS column:

- `wordform lemma pos id head deprel morph`

Columns are tab-separated. Sentence boundaries are expected to use `<s...>` and `</s>` lines.

In [1]:
# 1. Load the autoreload extension
%load_ext autoreload

# 2. Turn on automatic reloading
%autoreload 2

from pathlib import Path

from dep_colloc.depcolloc import (
    calculate_colloc_column_frequencies,
    filter_colloc_counts_by_vocabs,
    explode_colloc_counts,
    filter_vocab_by_frequency,
    generate_colloc_counts,
    switch_colloc_columns,
)
from dep_colloc.freq import gen_lemma_freq
from dep_colloc.matrix import create_raw_colloc_matrix
from dep_colloc.ppmi import (
    calculate_ppmi_dataframe,
    calculate_ppmi_sparse_file,
)
from dep_colloc.utils import (
    is_sentence_end,
    is_sentence_start,
    parse_token_line,
    pattern as TOKEN_PATTERN,
)

# Edit these paths for the corpus/output you want to process.
corpus_dir = Path('/home/volt/bach/Corpora/deu_news_1995-2025/deu_news_1995-2025_sentences_parsed/2021-2025')
output_dir = Path('/home/volt/bach/Corpora/deu_news_1995-2025/deu_news_1995-2025_lemma_depw2v/2021-2025/data_prep')
output_dir.mkdir(parents=True, exist_ok=True)

# Edit these paths for the corpus/output you want to process.
corpus_dir = Path('/home/volt/bach/Embeddings/type_embeddings/test_data')
output_dir = Path('/home/volt/bach/Embeddings/type_embeddings/test_output')
output_dir.mkdir(parents=True, exist_ok=True)

max_depth = 1
file_ext = '.txt'
freq_mode = 'lemma/pos'  # one of: token_only, lemma_only, token/pos, lemma/pos

# Format modes: token_only, lemma_only, token/pos, lemma/pos
target_format = 'lemma_only'
context_format = 'lemma_only'
append_deprel_path = True
colloc_output_filename = 'syn_colloc_counts.txt' if append_deprel_path else 'path_colloc_counts.txt'
malformed_log_path = output_dir / 'malformed_sentences.log'

## 2. Optional Depth QA

Use this section to inspect dependency-tree depth before generating collocation counts.

In [ ]:
from collections import Counter, defaultdict

from tqdm import tqdm


def build_dependency_tree(tokens, token_pattern=TOKEN_PATTERN):
    tree = defaultdict(list)
    roots = []
    for token in tokens:
        parsed = parse_token_line(token, token_pattern)
        if not parsed:
            continue
        if parsed.head == '0':
            roots.append(parsed.idx)
        else:
            tree[parsed.head].append(parsed.idx)
    return tree, roots


def get_max_depth(tree, root):
    visited = set()
    stack = [(root, 1)]
    max_depth_seen = 1

    while stack:
        node, depth = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        max_depth_seen = max(max_depth_seen, depth)
        for child in tree.get(node, []):
            stack.append((child, depth + 1))

    return max_depth_seen


def analyze_max_depth_distribution(corpus_path, token_pattern=TOKEN_PATTERN):
    depths = []
    files = sorted(Path(corpus_path).glob('*.txt'))

    for path in tqdm(files, desc='Analyzing syntactic depths'):
        sentence = []
        with path.open(encoding='utf-8') as input_file:
            for raw_line in input_file:
                line = raw_line.strip()
                if is_sentence_start(line):
                    sentence = []
                elif is_sentence_end(line):
                    tree, roots = build_dependency_tree(sentence, token_pattern)
                    depths.extend(get_max_depth(tree, root) for root in roots)
                elif line:
                    sentence.append(line)
    return depths


def plot_depth_distribution(depths, save_path=None, min_count=1):
    import matplotlib.pyplot as plt
    depth_counter = Counter(depths)
    filtered = [depth for depth in depths if depth_counter[depth] >= min_count]
    if not filtered:
        print(f'No depths with frequency >= {min_count}')
        return

    plt.figure(figsize=(10, 6))
    plt.hist(filtered, bins=range(1, max(filtered) + 2), edgecolor='black')
    plt.title(f'Distribution of Max Syntactic Depth (min_count >= {min_count})')
    plt.xlabel('Max Depth')
    plt.ylabel('Number of Sentences')
    plt.grid(True)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def print_files_with_depth_threshold(corpus_path, target_depth, token_pattern=TOKEN_PATTERN):
    matching_files = set()
    files = sorted(Path(corpus_path).glob('*.txt'))

    for path in tqdm(files, desc=f'Searching for depth >= {target_depth}'):
        sentence = []
        with path.open(encoding='utf-8') as input_file:
            for raw_line in input_file:
                line = raw_line.strip()
                if is_sentence_start(line):
                    sentence = []
                elif is_sentence_end(line):
                    tree, roots = build_dependency_tree(sentence, token_pattern)
                    if any(get_max_depth(tree, root) >= target_depth for root in roots):
                        matching_files.add(path.name)
                elif line:
                    sentence.append(line)

    for filename in sorted(matching_files):
        print(filename)
    return sorted(matching_files)

In [ ]:
# Optional depth QA
# depths = analyze_max_depth_distribution(corpus_dir)
# plot_depth_distribution(depths, save_path=output_dir / 'depth_distribution.png', min_count=10)
# deep_files = print_files_with_depth_threshold(corpus_dir, target_depth=10)

## 3. Frequency Counts

Generate lemma/token frequency files used later for PPMI weighting.

In [ ]:
freq_path = gen_lemma_freq(
    corpus_path=str(corpus_dir),
    out_folder=str(output_dir),
    file_ext=file_ext,
    mode=freq_mode,
)
print(f'Saved frequencies to {freq_path}')

In [ ]:
def count_total_value(filepath):
    total_value = 0
    with open(filepath, encoding='utf-8') as input_file:
        for line in input_file:
            if not line.strip():
                continue
            try:
                _, count_text = line.rstrip().rsplit('	', 1)
                total_value += int(count_text)
            except ValueError:
                print(f'Skipping malformed line: {line.rstrip()}')
    return total_value


total_tokens = count_total_value(output_dir / f"{freq_mode.replace('/', '_')}_freq.txt")
print(f'Total counted tokens: {total_tokens}')

## 4. Collocation Counts

`generate_colloc_counts(...)` writes ordered target/context co-occurrence within dependency distance `max_depth`. Target and context can use independent formats: `token_only`, `lemma_only`, `token/pos`, or `lemma/pos`.

When `append_deprel_path=False`, the context is just the formatted context item. When `append_deprel_path=True`, the same traversal appends the directed dependency path to the context as `formatted_context/directed_relation_path`, for example `run/pa_nsubj>chi_obj`.

Processing stops at the first malformed dependency sentence. Its source file, sentence ID, and validation error are written to `malformed_log_path`.

In [2]:
generate_colloc_counts(
    corpus_dir=str(corpus_dir),
    output_dir=str(output_dir),
    max_depth=max_depth,
    pattern=TOKEN_PATTERN,
    output_filename=colloc_output_filename,
    target_format=target_format,
    context_format=context_format,
    append_deprel_path=append_deprel_path,
    malformed_log_path=str(malformed_log_path),
)

Colloc files: 100%|██████████| 1/1 [00:00<00:00, 1261.44it/s]

Writing colloc counts to: /home/volt/bach/Embeddings/type_embeddings/test_output/syn_colloc_counts.txt


'/home/volt/bach/Embeddings/type_embeddings/test_output/syn_colloc_counts.txt'

## 4a. Reverse Pair Counts and/or Generate Vocabularies

Choose any pair-count file with `target context<TAB>count`. First compute `wvocab` from the first column and `cvocab` from the second column. Then reverse the same pair-count file into `context target<TAB>count` for training.

In [4]:
pair_counts_for_vocab = output_dir / 'syn_colloc_counts_reversed.txt'
wvocab_output = output_dir / 'wvocab.txt'
cvocab_output = output_dir / 'cvocab.txt'

wvocab_path, cvocab_path = calculate_colloc_column_frequencies(
    input_path=str(pair_counts_for_vocab),
    first_col_freq_path=str(wvocab_output),
    second_col_freq_path=str(cvocab_output),
)
print(f'Saved word vocabulary to {wvocab_path}')
print(f'Saved context vocabulary to {cvocab_path}')


Saved word vocabulary to /home/volt/bach/Embeddings/type_embeddings/test_output/wvocab.txt
Saved context vocabulary to /home/volt/bach/Embeddings/type_embeddings/test_output/cvocab.txt


In [3]:
pair_counts_for_reverse = output_dir / 'syn_colloc_counts.txt'
reversed_colloc_output = output_dir / 'syn_colloc_counts_reversed.txt'

reversed_path = switch_colloc_columns(
    input_path=str(pair_counts_for_reverse),
    switched_output_path=str(reversed_colloc_output),
)
print(f'Saved reversed pair counts to {reversed_path}')


Saved reversed pair counts to /home/volt/bach/Embeddings/type_embeddings/test_output/syn_colloc_counts_reversed.txt


## 4b. Filter Pair Counts By Vocabulary Frequency

Choose the pair-count file, `wvocab`, and `cvocab` paths explicitly. This section applies one shared frequency threshold to both vocabularies, filters the pair-count file using those filtered vocabularies, then recomputes final `wvocab_filtered` and `cvocab_filtered` from the filtered pairs.

In [5]:
# Section 4b: filter any pair-count file using user-specified vocab files.
min_frequency = 1

wvocab_for_filter = output_dir / 'wvocab.txt'
wvocab_filtered_output = output_dir / 'wvocab_filtered.txt'

cvocab_for_filter = output_dir / 'cvocab.txt'
cvocab_filtered_output = output_dir / 'cvocab_filtered.txt'

pair_counts_for_filter = output_dir / 'syn_colloc_counts_reversed.txt'
pair_counts_filtered_output = output_dir / 'syn_colloc_counts_reversed_filtered.txt'


In [6]:
wvocab_thresholded_path = filter_vocab_by_frequency(
    min_frequency=min_frequency,
    vocab_path=str(wvocab_for_filter),
    vocab_filtered_path=str(wvocab_filtered_output),
)
cvocab_thresholded_path = filter_vocab_by_frequency(
    min_frequency=min_frequency,
    vocab_path=str(cvocab_for_filter),
    vocab_filtered_path=str(cvocab_filtered_output),
)
pair_counts_filtered_path = filter_colloc_counts_by_vocabs(
    syn_colloc_counts_path=str(pair_counts_for_filter),
    cvocab_path=cvocab_thresholded_path,
    wvocab_path=wvocab_thresholded_path,
    syn_colloc_counts_filtered_path=str(pair_counts_filtered_output),
)
wvocab_filtered_path, cvocab_filtered_path = calculate_colloc_column_frequencies(
    input_path=pair_counts_filtered_path,
    first_col_freq_path=str(wvocab_filtered_output),
    second_col_freq_path=str(cvocab_filtered_output),
)
print(f'Saved thresholded word vocabulary to {wvocab_thresholded_path}')
print(f'Saved thresholded context vocabulary to {cvocab_thresholded_path}')
print(f'Saved filtered pair counts to {pair_counts_filtered_path}')
print(f'Saved final word vocabulary to {wvocab_filtered_path}')
print(f'Saved final context vocabulary to {cvocab_filtered_path}')

Saved thresholded word vocabulary to /home/volt/bach/Embeddings/type_embeddings/test_output/wvocab_filtered.txt
Saved thresholded context vocabulary to /home/volt/bach/Embeddings/type_embeddings/test_output/cvocab_filtered.txt
Saved filtered pair counts to /home/volt/bach/Embeddings/type_embeddings/test_output/syn_colloc_counts_reversed_filtered.txt
Saved final word vocabulary to /home/volt/bach/Embeddings/type_embeddings/test_output/wvocab_filtered.txt
Saved final context vocabulary to /home/volt/bach/Embeddings/type_embeddings/test_output/cvocab_filtered.txt


## 4c. Explode Pair Counts For word2vecf

This `word2vecf` version reads two whitespace-separated tokens per training event, so a pair-count file such as `context target<TAB>count` must be expanded into repeated `context target` lines. Choose the pair-count file with `pair_counts_for_explode`.

In [7]:
pair_counts_for_explode = output_dir / 'syn_colloc_counts_reversed_filtered.txt'
dep_contexts_output = output_dir / 'dep.contexts'

dep_contexts_path = explode_colloc_counts(
    input_path=str(pair_counts_for_explode),
    output_path=str(dep_contexts_output),
)
print(f'Saved exploded dependency contexts to {dep_contexts_path}')


Saved exploded dependency contexts to /home/volt/bach/Embeddings/type_embeddings/test_output/dep.contexts


## 5. Create A Raw Dependency Collocation Matrix

Create a target-by-context count matrix from either `syn_colloc_counts.txt` or `path_colloc_counts.txt`.

Choose `dataframe` for interactive work on a manageable matrix, `sparse_npz` for a compact SciPy CSR matrix, or `matrix_market` to stream a very large sparse matrix to disk without constructing the full matrix in memory. Saved sparse formats include separate row and column label files.

In [14]:
# Input can be syn_colloc_counts.txt or path_colloc_counts.txt.
pair_counts_for_matrix = output_dir / 'syn_colloc_counts_reversed.txt'
raw_matrix_output_prefix = output_dir / 'syn_colloc_raw_matrix'
raw_matrix_format = 'dataframe'  # dataframe, sparse_npz, or matrix_market

raw_colloc_matrix = create_raw_colloc_matrix(
    input_path=str(pair_counts_for_matrix),
    output_prefix=str(raw_matrix_output_prefix),
    output_format=raw_matrix_format,
)

if raw_matrix_format == 'dataframe':
    print(f'Raw sparse DataFrame shape: {raw_colloc_matrix.shape}')
else:
    raw_matrix_path, row_labels_path, column_labels_path = raw_colloc_matrix
    print(f'Saved raw matrix to {raw_matrix_path}')
    print(f'Saved row labels to {row_labels_path}')
    print(f'Saved column labels to {column_labels_path}')

Raw sparse DataFrame shape: (14, 8)


## 5a. Convert The Raw Count Matrix To PPMI

PPMI is calculated only for nonzero count cells using marginals from the raw target-by-context matrix. DataFrame input returns a pandas sparse DataFrame; disk-based input writes a compressed SciPy `.npz` matrix. The row and column label files from Section 5 apply unchanged to the PPMI matrix.

In [ ]:
ppmi_min_count = 1
ppmi_output_path = output_dir / 'syn_colloc_ppmi_matrix.npz'

if raw_matrix_format == 'dataframe':
    ppmi_matrix = calculate_ppmi_dataframe(
        raw_colloc_matrix,
        min_count=ppmi_min_count,
    )
    print(f'PPMI sparse DataFrame shape: {ppmi_matrix.shape}')
else:
    ppmi_matrix_path = calculate_ppmi_sparse_file(
        count_matrix_path=raw_matrix_path,
        output_path=str(ppmi_output_path),
        min_count=ppmi_min_count,
    )
    print(f'Saved sparse PPMI matrix to {ppmi_matrix_path}')

## 6. Export Helpers

These helpers convert matrix outputs into formats needed by downstream tools. Pair-count expansion for `word2vecf` is handled in section 4c.


In [ ]:
def save_df_to_pac(df_path, output_path):
    import os
    import sys

    import pandas as pd
    from scipy.sparse import coo_matrix

    sys.path.append('/home/volt/bach/KUL/nephosem')
    sys.path.append('/home/volt/bach/KUL/semasioFlow')
    from nephosem import TypeTokenMatrix

    output_path = str(output_path)
    if output_path.endswith('/'):
        output_path = os.path.join(output_path, 'colloc_matrix')
    if not output_path.endswith('.pac'):
        output_path += '.pac'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    df = pd.read_csv(df_path, index_col=0)
    sparse_csr = coo_matrix(df.values).tocsr()
    pac = TypeTokenMatrix(
        matrix=sparse_csr,
        row_items=df.index.tolist(),
        col_items=df.columns.tolist(),
    )
    pac.save(output_path)
    print(f'Saved .pac to: {output_path}')

In [ ]:
# Optional exports
# save_df_to_pac(
#     df_path=str(colloc_matrix_csv),
#     output_path=str(output_dir / 'colloc_matrix.pac'),
# )


## 7. Debug Comparison Helper

Keep ad hoc file comparisons here so they do not interrupt the main workflow.

In [ ]:
from collections import Counter


def read_lines(path):
    with open(path, encoding='utf-8') as input_file:
        return Counter(line.rstrip('\n') for line in input_file if line.strip())


def compare_files(path1, path2):
    counter1 = read_lines(path1)
    counter2 = read_lines(path2)

    only1 = sorted(set(counter1) - set(counter2))
    only2 = sorted(set(counter2) - set(counter1))
    common = sorted(set(counter1) & set(counter2))

    print(f'Common lines: {len(common)}')
    print(f'Only in {path1}: {len(only1)}')
    print(f'Only in {path2}: {len(only2)}')
    return {'common': common, 'only1': only1, 'only2': only2}